
# HODGE v10a.31 — M5–M7 MATH-ONLY A100 Runner

This runner performs one requested coefficient calculation and nothing else.

It does **not** execute the v10a.26 fourth-order notebook, an M4 census, lower-order regression locks, synthetic SW tests, GPU/CPU comparisons, one-face smoke runs, duplicate-rotation recomputations, reverse-Hermiticity audit runs, or an exhaustive unused Haar-cap pre-census.

For the selected order it performs only:

\[
\text{requested-order support census}
\rightarrow
\text{rooted connected support closure}
\rightarrow
\text{physical Gram/Haar contractions actually encountered}
\rightarrow
\text{one-flux and vacuum Krylov models}
\rightarrow
\text{SW/BCH coefficient}
\rightarrow
\text{rooted incidence subtraction}.
\]

Set `ORDER = 5` for the first run. Change it to `6` or `7` for later runs.


In [ ]:

from __future__ import annotations

import ast
import json
import os
import shutil
import symtable
import time
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import opt_einsum as oe
import sympy as sp

# ------------------------------ RUN SETTINGS ------------------------------
ORDER = 5                       # allowed: 5, 6, 7
USE_GOOGLE_DRIVE = True
WORKDIR_NAME = "HODGE_M5_M7_MATH_ONLY"
AUTO_UPLOAD_MISSING_FILES = True

V26_NAME = "NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb"
V28_NAME = "ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py"

MAX_NEW_SHAPES = 0             # 0 = all remaining shapes
TIME_BUDGET_MINUTES = 0        # 0 = no between-shape stop
GPU_SW_MIN_DIM = 64
HEARTBEAT_SECONDS = 20

if ORDER not in (5, 6, 7):
    raise ValueError("ORDER must be 5, 6, or 7")


In [ ]:

# Durable checkpoints and source-file loading. No physics is run in this cell.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORKDIR = Path("/content/drive/MyDrive") / WORKDIR_NAME
else:
    WORKDIR = Path("/content") / WORKDIR_NAME
WORKDIR.mkdir(parents=True, exist_ok=True)

CONTENT = Path("/content")

def locate(name: str) -> Path | None:
    for p in (CONTENT / name, WORKDIR / name, Path.cwd() / name):
        if p.exists():
            return p
    return None

def upload_if_missing(name: str) -> Path:
    p = locate(name)
    if p is not None:
        return p
    if not AUTO_UPLOAD_MISSING_FILES:
        raise FileNotFoundError(name)
    from google.colab import files
    print(f"Upload {name}")
    uploaded = files.upload()
    if name in uploaded:
        return CONTENT / name
    if len(uploaded) == 1:
        src = CONTENT / next(iter(uploaded))
        dst = CONTENT / name
        if src != dst:
            shutil.move(str(src), str(dst))
        return dst
    raise RuntimeError(f"Expected {name}; received {list(uploaded)}")

V26_PATH = upload_if_missing(V26_NAME)
V28_PATH = upload_if_missing(V28_NAME)
ORDER_DIR = WORKDIR / f"m{ORDER}"
ORDER_DIR.mkdir(parents=True, exist_ok=True)
print("ORDER:", ORDER)
print("WORKDIR:", WORKDIR)


In [ ]:

# Extract only the definitions and initializers required by the actual M5–M7 engine.
# Known v10a.26 M4 production statements are rejected rather than executed.
V26_ROOTS = (
    "L", "N", "faces", "verts", "T1_POLS", "anchor_faces", "V23C_ROOT",
    "V23C_POL", "_FAST_EPS", "oe", "LXState", "_V17_VAC",
    "_v17_apply_W_faces", "_v17_apply_W_labeled", "_v17_connected",
    "_v17_phys_index", "_v17_translate_support", "_v17_translate_face",
    "_v23c_split_h0", "_v23c_rooted_connected_subsets", "_v24c_shape_key",
    "_v24c_candidate_supports", "_v10a3_face_state",
    "_v10a3_physical_blocks", "_v10a3_compress_state", "_v9_flux_key_state",
    "_joint_canon_states", "lx_combine_bra_ket",
    "_v26_singlet_multiplicity",
)

_MUTATING = {"add", "append", "clear", "discard", "extend", "insert", "pop", "remove", "setdefault", "sort", "update"}

def _base_name(node):
    while isinstance(node, (ast.Attribute, ast.Subscript)):
        node = node.value
    return node.id if isinstance(node, ast.Name) else None

def _defined_or_mutated(stmt):
    names = set()
    class V(ast.NodeVisitor):
        def visit_FunctionDef(self, node): names.add(node.name)
        visit_AsyncFunctionDef = visit_FunctionDef
        def visit_ClassDef(self, node): names.add(node.name)
        def visit_Import(self, node):
            for a in node.names: names.add(a.asname or a.name.split(".")[0])
        def visit_ImportFrom(self, node):
            for a in node.names:
                if a.name != "*": names.add(a.asname or a.name)
        def visit_Name(self, node):
            if isinstance(node.ctx, (ast.Store, ast.Del)): names.add(node.id)
        def visit_Call(self, node):
            if isinstance(node.func, ast.Attribute) and node.func.attr in _MUTATING:
                base = _base_name(node.func.value)
                if base: names.add(base)
            self.generic_visit(node)
    V().visit(stmt)
    return names

def _dependencies(stmt, all_names):
    table = symtable.symtable(ast.unparse(stmt), "<bootstrap>", "exec")
    deps = set()
    def walk(tab):
        module = tab.get_type() == "module"
        for sym in tab.get_symbols():
            if sym.is_referenced() and (module or sym.is_global()):
                deps.add(sym.get_name())
        for child in tab.get_children(): walk(child)
    walk(table)
    return deps & all_names

def build_definitions_only_source(path: Path) -> str:
    doc = nbformat.read(path, as_version=4)
    source = "\n\n".join(c.source for c in doc.cells if c.cell_type == "code" and c.source.strip())
    statements = list(ast.parse(source, filename=str(path)).body)
    defines, defmap = [], {}
    for i, stmt in enumerate(statements):
        dn = _defined_or_mutated(stmt)
        defines.append(dn)
        for name in dn: defmap.setdefault(name, set()).add(i)

    missing = [name for name in V26_ROOTS if name not in defmap]
    if missing:
        raise RuntimeError("v10a.26 is missing required definitions: " + ", ".join(missing))

    selected = {i for i, s in enumerate(statements) if isinstance(s, (ast.Import, ast.ImportFrom))}
    pending, resolved = list(V26_ROOTS), set()
    all_names = set(defmap)
    while pending:
        name = pending.pop()
        if name in resolved: continue
        resolved.add(name)
        for idx in sorted(defmap.get(name, ())):
            selected.add(idx)
            for dep in _dependencies(statements[idx], all_names):
                if dep not in resolved: pending.append(dep)

    forbidden = (
        "FULL-T1 OPERATOR MOMENTS", "N=<R1|R1>", "J=<R1|R^2B>",
        "C1=<R1|RWRB>", "D=<WRB|RWRB>", "ROOTED INCIDENCE TRANSFORM",
        "shape_cache = _v26_load_checkpoint", "shape_cache=_v26_load_checkpoint",
        "for ci, C in enumerate(CLUST", "for ci,C in enumerate(CLUST",
        "_v23c_fit_cluster(preC", "v10a.26 preflight: one-face Q2",
    )
    rejected = []
    for idx in sorted(selected):
        stmt = statements[idx]
        if isinstance(stmt, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            continue
        segment = ast.get_source_segment(source, stmt) or ast.unparse(stmt)
        hit = next((x for x in forbidden if x in segment), None)
        if hit: rejected.append((idx, hit))
    if rejected:
        raise RuntimeError(f"M4 production leaked into definitions bootstrap: {rejected}")

    module = ast.fix_missing_locations(ast.Module(body=[statements[i] for i in sorted(selected)], type_ignores=[]))
    return ast.unparse(module) + "\n"

os.environ["PREFER_GPU"] = "1"
os.environ["GLUE_L"] = "5"
os.environ["V10A23_CLUSTER_POL"] = "2"
os.environ["V10A23_CLUSTER_PROGRESS"] = "0"
os.environ["V10A7_SUPPORT_POLS"] = "2"
os.environ["V10A7_RECHECK_Q1"] = "0"
os.environ["V10A7_UNBLIND"] = "0"

exec(compile(build_definitions_only_source(V26_PATH), "<v26-definitions-only>", "exec"), globals())
missing = [name for name in V26_ROOTS if name not in globals()]
if missing:
    raise RuntimeError("Definitions bootstrap failed: " + ", ".join(missing))
for forbidden_name in ("Dop", "K4op", "V26_RESULT", "M4_ORACLE", "shape_cache"):
    if forbidden_name in globals():
        raise RuntimeError(f"Forbidden M4 state exists: {forbidden_name}")
print("Definitions loaded. No v10a.26 M4 calculation executed.")


In [ ]:

# Convert v10a.28 into a requested-order-only calculation engine.
def patch_math_only(source: str) -> str:
    original = source

    required = (
        'V28_SCHEMA = "hodge-v10a28-order-aware-krylov-gram-haar9-v1"',
        'print("\\n[1] FACTORIZED SU(3) HAAR CERTIFICATE")',
        '# 3. Exact-SW arbitrary-order regression',
        '# 4. Generic bidirectional support history census',
        'if V28_RUN_CENSUS:',
        '# 5. Checkpointed production and rooted incidence transform',
        'def _v28_lower_targets():',
        'print("\\n[5] PREFLIGHT SUMMARY")',
    )
    missing = [x for x in required if x not in source]
    if missing:
        raise RuntimeError(f"Unexpected v10a.28 source; missing patch anchors: {missing}")

    source = source.replace(
        'V28_SCHEMA = "hodge-v10a28-order-aware-krylov-gram-haar9-v1"',
        'V28_SCHEMA = "hodge-v10a31-math-only-v1"', 1,
    )
    source = source.replace(
        'if V28_ORDER not in (4, 5, 6, 7):\n    raise ValueError("V28_ORDER must be 4, 5, 6, or 7")',
        'if V28_ORDER not in (5, 6, 7):\n    raise ValueError("V28_ORDER must be 5, 6, or 7")', 1,
    )
    source = source.replace(
        'if V28_HERMITICITY_AUDIT_PAIRS < 1:\n    raise ValueError("V28_HERMITICITY_AUDIT_PAIRS must be at least one")',
        'if V28_HERMITICITY_AUDIT_PAIRS < 0:\n    raise ValueError("V28_HERMITICITY_AUDIT_PAIRS must be nonnegative")', 1,
    )

    # Remove globals used only by the deleted regression/comparator code.
    source = source.replace(
        '    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",\n',
        '    "_v24c_candidate_supports", "_v10a3_face_state",\n', 1,
    )
    source = source.replace(
        '    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",\n'
        '    "_v23_sw_exact", "_v23_sp", "_v23_random", "_V23CF",\n'
        '    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",\n',
        '    "_joint_canon_states", "lx_combine_bra_ket",\n'
        '    "_v26_singlet_multiplicity",\n', 1,
    )

    # Remove gate framework/banner and replace it with a plain calculation banner.
    gate_start = source.index('V28_GATES = []')
    section1 = source.index('# ---------------------------------------------------------------------------\n# 1. Exact invariant-basis Haar projectors', gate_start)
    source = source[:gate_start] + (
        'print("=" * 110)\n'
        'print(f"HODGE v10a.31 -- M{V28_ORDER} MATH-ONLY PRODUCTION")\n'
        'print("=" * 110)\n'
        'print("Krylov depth:", V28_KRYLOV_DEPTH)\n'
        'print("support half-depth:", V28_SUPPORT_HALF_DEPTH)\n'
        'print("Haar occurrence cap:", V28_HAAR_CAP)\n'
        'print("backend:", "CuPy/CUDA" if V28_GPU_ENABLED else "CPU")\n\n'
    ) + source[section1:]

    # A projector is built only when an actual contraction encounters its pattern.
    # Remove the separate exhaustive cap certification and its unused helper.
    cert_func_start = source.index('def _v28_certify_haar_cap():')
    cert_func_end = source.index('@lru_cache(maxsize=V28_HAAR_CACHE_SIZE)', cert_func_start)
    source = source[:cert_func_start] + source[cert_func_end:]
    cert_call_start = source.index('print("\\n[1] FACTORIZED SU(3) HAAR CERTIFICATE")')
    section2 = source.index('# ---------------------------------------------------------------------------\n# 2. Order-generic physical Krylov basis', cert_call_start)
    source = source[:cert_call_start] + source[section2:]

    # Remove extra projector span/rank diagnostics after the actual inverse is constructed.
    proj_tail_start = source.index('    G = Graw[np.ix_(keep, keep)]\n    Gsp = sp.Matrix(G.tolist())')
    proj_tail_end = source.index('    return I, C, cert\n', proj_tail_start) + len('    return I, C, cert\n')
    minimal_tail = '''    G = Graw[np.ix_(keep, keep)]\n    Gsp = sp.Matrix(G.tolist())\n    C = np.asarray(Gsp.inv().tolist(), dtype=np.float64)\n    I = R[np.asarray(keep, dtype=int)].reshape((expected,) + (3,) * m).astype(np.float64)\n    return I, C, None\n'''
    source = source[:proj_tail_start] + minimal_tail + source[proj_tail_end:]

    # Remove every synthetic SW/BCH/GPU/band regression and the one-face smoke run.
    regression_start = source.index('# ---------------------------------------------------------------------------\n# 3. Exact-SW arbitrary-order regression')
    support_def_start = source.index('# ---------------------------------------------------------------------------\n# 4. Generic bidirectional support history census', regression_start)
    source = source[:regression_start] + source[support_def_start:]

    # Replace the entire top-level census/preflight block with the requested-order census only.
    census_start = source.index('if V28_RUN_CENSUS:')
    section5 = source.index('# ---------------------------------------------------------------------------\n# 5. Checkpointed production and rooted incidence transform', census_start)
    census_block = r'''if not V28_RUN_CENSUS:
    raise RuntimeError("math-only production requires the requested-order support census")

_v28_census_cache = _v28_load_census()
V28_O4_SUPPORTS, V28_O4_STATS = set(), {}
if _v28_census_cache is None:
    print(f"\nREQUESTED-ORDER SUPPORT CENSUS: M{V28_ORDER}")
    _v28_histories = _v28_history_levels(V28_SUPPORT_HALF_DEPTH, V23C_POL)
    V28_MAXC, V28_ENDPOINT_SUPPORTS, V28_SUPPORT_STATS, _v28_histories = (
        _v28_support_census(V28_ORDER, V23C_POL, _v28_histories)
    )
    _v28_save_census(set(), {}, V28_MAXC, V28_SUPPORT_STATS)
else:
    V28_MAXC = set(_v28_census_cache["maxc"])
    V28_ENDPOINT_SUPPORTS = set()
    V28_SUPPORT_STATS = _v28_census_cache["support_stats"]
    print(f"\nLoaded M{V28_ORDER} support census: {len(V28_MAXC):,} supports")

if not V28_MAXC:
    raise RuntimeError(f"M{V28_ORDER} support census is empty")

_extent_rows = [(C,) + _v28_rooted_extent(C) for C in V28_MAXC]
_boundary = [row for row in _extent_rows if row[2]]
V28_MAX_EXTENT = max((row[1] for row in _extent_rows), default=0)
if _boundary:
    raise RuntimeError(
        f"periodic support alias at L={L}: {len(_boundary)} supports touch the half-box boundary"
    )

V28_CLUSTERS = set()
for C in V28_MAXC:
    V28_CLUSTERS.update(_v23c_rooted_connected_subsets(C))
if not V28_CLUSTERS:
    raise RuntimeError(f"M{V28_ORDER} rooted support closure is empty")
V28_SHAPE_KEYS = {_v24c_shape_key(C) for C in V28_CLUSTERS}
print("maximal supports:", len(V28_MAXC))
print("rooted clusters:", len(V28_CLUSTERS))
print("rotation classes:", len(V28_SHAPE_KEYS))
print("size histogram:", dict(sorted(Counter(map(len, V28_CLUSTERS)).items())))

'''
    source = source[:census_start] + census_block + source[section5:]

    # No independent reverse-direction Hermiticity sampling; W is assembled once and symmetrized.
    audit_start = source.index('    # Independent reverse-direction checks: W is filled symmetrically, so the')
    audit_end = source.index('    counts = tuple(len(x) for x in layers)', audit_start)
    source = source[:audit_start] + '    herm = 0.0\n    audit_pairs = ()\n' + source[audit_end:]

    # Remove lower-order target data entirely.
    lower_start = source.index('def _v28_lower_targets():')
    prod_start = source.index('def _v28_production():', lower_start)
    source = source[:lower_start] + source[prod_start:]
    source = source.replace(
        'def _v28_production():\n    if not all(ok for _, ok, _ in V28_GATES):\n'
        '        raise RuntimeError("v10a.28 preflight gate failure; production is blocked")\n'
        '    cache = _v28_load()\n',
        'def _v28_production():\n    cache = _v28_load()\n', 1,
    )

    # Remove duplicate proper-rotation recomputation and retain only c_n per cluster.
    dup_start = source.index('            if duplicate_checks < V28_DUPLICATE_CHECKS and C != representatives[key]:')
    raw_line = source.index('            raw[C] = np.asarray(item["coef"], dtype=np.float64).copy()', dup_start)
    raw_end = raw_line + len('            raw[C] = np.asarray(item["coef"], dtype=np.float64).copy()')
    source = source[:dup_start] + '            raw[C] = float(item["coef"][V28_ORDER])' + source[raw_end:]

    # Replace post-run gates and the all-lower-order rooted ledger with only m_n.
    post_start = source.index('    v28_gate(\n        f"exact SW block diagonalization closes through O(u^{V28_ORDER})"')
    omega_start = source.index('    omega = {}', post_start)
    return_end = source.index('    return result\n', omega_start) + len('    return result\n')
    production_tail = r'''    if max_offdiag >= V28_SW_TOL:
        raise RuntimeError(f"SW P-Q residual is {max_offdiag:.3e}")
    if len(cache) != len(V28_SHAPE_KEYS):
        raise RuntimeError(f"missing shape coefficients: {len(cache)}/{len(V28_SHAPE_KEYS)}")

    omega = {}
    total = 0.0
    by_size = defaultdict(float)
    for C in sorted(V28_CLUSTERS, key=lambda x: (len(x), tuple(sorted(x)))):
        z = float(raw[C])
        for S in _v23c_rooted_connected_subsets(C):
            if S != C:
                z -= omega[S]
        omega[C] = z
        total += z
        by_size[len(C)] += z

    print("\nROOTED INCIDENCE TRANSFORM")
    for size in sorted(by_size):
        print(f"  size {size}: m{V28_ORDER}={by_size[size]:+.12g}")
    print(f"  TOTAL: m{V28_ORDER}={total:+.15g}")

    result = dict(
        schema=V28_SCHEMA, signature=V28_RUN_SIGNATURE, order=V28_ORDER,
        coefficient=float(total), by_size=dict(by_size), omega=omega,
        concrete_clusters=len(V28_CLUSTERS), shapes=len(V28_SHAPE_KEYS),
        haar_cap=V28_HAAR_CAP, krylov_depth=V28_KRYLOV_DEPTH,
        target_loaded=False,
    )
    print("\nBLIND PRODUCTION RESULT")
    print(f"  m{V28_ORDER} = {total!r}")
    return result
'''
    source = source[:post_start] + production_tail + source[return_end:]

    # No preflight summary or firewall mode. Execute the requested calculation directly.
    footer_start = source.index('print("\\n[5] PREFLIGHT SUMMARY")')
    source = source[:footer_start] + 'V28_RESULT = _v28_production()\n'

    banned_calls = {
        "_v28_certify_haar_cap", "_v28_sw_regression", "_v28_gpu_sw_regression",
        "_v28_order_band_regression", "_v23c_fit_cluster", "v28_gate",
    }
    tree = ast.parse(source)
    seen = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name): seen.add(node.func.id)
            elif isinstance(node.func, ast.Attribute): seen.add(node.func.attr)
    bad = sorted(banned_calls & seen)
    if bad:
        raise RuntimeError(f"non-production calls remain after patch: {bad}")

    forbidden_text = (
        "GENERIC SUPPORT-CENSUS REGRESSION AT ORDER FOUR",
        "ONE-FACE PHYSICAL PREFIX REGRESSION",
        "ONE-FACE PHYSICAL SMOKE",
        "PREFLIGHT SUMMARY",
        "Preserve unconditional lower-order regression coverage",
        "-0.7751458630189173",
        "through O4--O7",
    )
    leftovers = [x for x in forbidden_text if x in source]
    if leftovers:
        raise RuntimeError(f"non-production text remains after patch: {leftovers}")
    if source == original:
        raise RuntimeError("math-only patch made no changes")
    return source

V28_SOURCE = patch_math_only(V28_PATH.read_text(encoding="utf-8", errors="strict"))
print("Engine stripped to requested-order production only.")


In [ ]:

# Run the selected order. This is the first and only physics calculation.
cap = 7 if ORDER == 5 else 9
os.environ.update({
    "V28_ORDER": str(ORDER),
    "V28_MODE": "production",
    "V28_HAAR_CAP": str(cap),
    "V28_RUN_CENSUS": "1",
    "V28_GPU": "1",
    "V28_GPU_SW_MIN_DIM": str(GPU_SW_MIN_DIM),
    "V28_HERMITICITY_AUDIT_PAIRS": "0",
    "V28_DUPLICATE_CHECKS": "0",
    "V28_HEARTBEAT": str(HEARTBEAT_SECONDS),
    "V28_RESUME": "1",
    "V28_PRODUCTION_CONFIRM": f"YES_ORDER_{ORDER}",
    "V28_MAX_NEW_SHAPES": str(MAX_NEW_SHAPES),
    "V28_TIME_BUDGET_MINUTES": str(TIME_BUDGET_MINUTES),
    "V28_CHECKPOINT": str(ORDER_DIR / "shapes.pkl"),
    "V28_CENSUS_CHECKPOINT": str(ORDER_DIR / "census.pkl"),
})

for name in tuple(globals()):
    if name.startswith("V28_") and name not in {"V28_PATH", "V28_SOURCE"}:
        globals().pop(name, None)

t0 = time.time()
exec(compile(V28_SOURCE, "<v10a31-math-only-engine>", "exec"), globals())
elapsed = time.time() - t0
RESULT = globals().get("V28_RESULT")
if RESULT is None:
    raise RuntimeError("calculation returned no result")
print(f"Elapsed: {elapsed / 3600:.3f} h")
print("Complete:", RESULT.get("complete", True))


In [ ]:

# Save only the numerical result and resume locations.
if RESULT.get("complete", True):
    value = float(RESULT["coefficient"])
    payload = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": ORDER,
        f"m{ORDER}": value,
        "concrete_clusters": int(RESULT.get("concrete_clusters", 0)),
        "rotation_classes": int(RESULT.get("shapes", 0)),
        "krylov_depth": int(RESULT.get("krylov_depth", 0)),
        "haar_cap": int(RESULT.get("haar_cap", 0)),
        "elapsed_seconds": elapsed,
        "shape_checkpoint": os.environ["V28_CHECKPOINT"],
        "census_checkpoint": os.environ["V28_CENSUS_CHECKPOINT"],
        "external_target_loaded": False,
    }
    result_path = ORDER_DIR / f"blind_m{ORDER}_result.json"
    result_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"m{ORDER} =", repr(value))
    print("Result:", result_path)
else:
    print("Checkpoint saved. Rerun the calculation cell to continue.")
